<a href="https://colab.research.google.com/github/yiyu-chen-labs/llm-from-scratch/blob/main/day24-sft-dataset/build_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/sakana_day24/"
os.makedirs(BASE, exist_ok=True)
print(BASE)

In [ ]:
import random, json, os
from datetime import date, timedelta

random.seed(42)

企業 = ["営業部", "開発部", "経営企画部", "人事部", "マーケティング部", "法務部"]
姓 = ["田中", "佐藤", "鈴木", "高橋", "伊藤", "渡辺", "山本", "中村", "小林", "加藤"]
議題 = ["新製品ローンチ", "第3四半期予算", "採用計画", "システム移行",
        "顧客満足度調査", "海外展開", "セキュリティ監査", "業務効率化"]
決定パターン = [
    "{x}を承認し、来月から実施する", "{x}については再検討とする",
    "{x}の予算を{n}万円に確定", "{x}を次回会議まで保留",
    "{x}の担当を{dept}に移管する",
]
タスクパターン = [
    "{x}の資料作成", "{x}のベンダー選定", "{x}の見積もり取得",
    "{x}の社内調整", "{x}の進捗レポート提出", "{x}の要件定義",
]

def gen_one():
    base = date(2026, random.randint(1, 12), random.randint(1, 28))
    topic = random.choice(議題)
    n_att = random.randint(3, 6)
    attendees = [f"{random.choice(姓)}（{random.choice(企業)}）" for _ in range(n_att)]
    attendees = list(dict.fromkeys(attendees))
    decisions = [
        random.choice(決定パターン).format(
            x=random.choice(議題), n=random.choice([50, 100, 300, 500, 1200]),
            dept=random.choice(企業))
        for _ in range(random.randint(1, 3))
    ]
    action_items = []
    for _ in range(random.randint(1, 4)):
        has_due = random.random() > 0.3
        due = (base + timedelta(days=random.randint(3, 45))).isoformat() if has_due else None
        action_items.append({
            "task": random.choice(タスクパターン).format(x=random.choice(議題)),
            "owner": random.choice(attendees).split("（")[0],
            "due": due,
        })
    return {
        "meeting_title": f"{topic}に関する定例会議",
        "date": base.isoformat(),
        "attendees": attendees,
        "decisions": decisions,
        "action_items": action_items,
        "next_meeting": (base + timedelta(days=random.choice([7, 14, 30]))).isoformat()
                        if random.random() > 0.25 else None,
    }

REC_PATH = BASE + "records.json"
if os.path.exists(REC_PATH):
    with open(REC_PATH, encoding="utf-8") as f:
        records = json.load(f)
    print(f"讀檔：{len(records)} 筆")
else:
    records = [gen_one() for _ in range(300)]
    with open(REC_PATH, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"新生成並存檔：{len(records)} 筆")

print(records[0]["date"])

In [ ]:
import re

PAIR_PATH = BASE + "pairs.json"
if os.path.exists(PAIR_PATH):
    with open(PAIR_PATH, encoding="utf-8") as f:
        pairs = {int(k): v for k, v in json.load(f).items()}
    print(f"讀檔：{len(pairs)} 筆")
else:
    pairs = {}
    print("新建 pairs")

def save_pairs():
    with open(PAIR_PATH, "w", encoding="utf-8") as f:
        json.dump(pairs, f, ensure_ascii=False, indent=2)

def ingest(raw):
    chunks = re.split(r'###(\d{4})###', raw)[1:]
    for i in range(0, len(chunks), 2):
        idx = int(chunks[i]) - 1
        pairs[idx] = {"note": chunks[i+1].strip(), "json": records[idx]}
    save_pairs()
    print(f"累計 {len(pairs)}/{len(records)}")

def status(n=20):
    missing = [i for i in range(len(records)) if i not in pairs]
    print(f"已完成 {len(pairs)}/{len(records)}")
    if not missing:
        print("全部完成"); return
    s = missing[0]
    print(f"缺 {len(missing)} 筆，下一批從 {s+1:04d} 開始\n" + "=" * 40)
    for offset, r in enumerate(records[s:s+n]):
        print(f"###{s+offset+1:04d}###")
        print(json.dumps(r, ensure_ascii=False))

In [ ]:
records[0]["date"]

In [ ]:
print(os.path.exists(REC_PATH))   # True
print(records[0]["date"])         # 2026-11-04

In [ ]:
status()


In [ ]:
ingest("""
###0001###
11月4日（水）　新製品ローンチ定例
出席：高橋（開発）、鈴木・佐藤（法務）、小林（営業）、加藤（人事）

第3四半期予算は承認され、来月から実施に移ることで決定した。大きな異論はなく、比較的スムーズにまとまった。
鈴木さんはセキュリティ監査の要件定義を12月18日までに整理する。
年内の残り期間が短いため、着手のタイミングだけ次回までに確認することにしました。
次回は2週間後の18日。

###0002###
1月25日（日）　採用計画定例
出席：渡辺（経営企画）、渡辺（営業）、渡辺（マーケティング）、鈴木（開発）、佐藤（人事）、佐藤（経営企画）

業務効率化は承認され、来月から実施に移る。第3四半期予算については次回会議まで保留とする。
渡辺さんはシステム移行の資料作成を2月1日までに、セキュリティ監査の資料作成を2月11日までに、海外展開のベンダー選定を2月20日までに進める。
渡辺さんが3名、佐藤さんが2名という構成のため、議事録では部署名を必ず併記します。休日開催だったこともあり、次回日程は改めて調整することにしました。

###0003###
5月23日（土）　第3四半期予算定例
出席：小林・伊藤（法務）、高橋（開発）、中村（人事）

第3四半期予算は100万円で確定した。システム移行は担当を営業部へ移管することで決定。新製品ローンチについては再検討とする。
伊藤さんはセキュリティ監査の社内調整を担当。中村さんはシステム移行のベンダー選定を進め、セキュリティ監査の進捗レポートを6月22日までに提出する。
移管先の営業部が不参加のため、引き継ぎは別途行います。次回の日程は未定。

###0004###
3月17日（火）　業務効率化定例
出席：田中（営業）、鈴木（法務）

第3四半期予算は担当を人事部へ、業務効率化は担当を経営企画部へそれぞれ移管することで決定した。
鈴木さんは顧客満足度調査の要件定義を3月27日までに整理する。
2名のみの開催となり、移管先の両部門も不在だったため、決定事項は後ほど関係者に共有します。次回日程も含めて改めて調整することに。

###0005###
7月6日（月）　業務効率化定例
出席：伊藤・鈴木（マーケティング）、佐藤（法務）

システム移行は担当を経営企画部へ移管することで決定。新製品ローンチについては再検討とする。
伊藤さんは採用計画の進捗レポートを今週13日までに提出し、顧客満足度調査の見積もり取得と第3四半期予算の進捗レポートもあわせて担当する。佐藤さんは顧客満足度調査のベンダー選定を8月13日までに進める。
伊藤さんに3件寄っているため、負荷は次回確認することにしました。
次回は7月20日。

###0006###
4月18日（土）　システム移行定例
出席：山本（法務）、渡辺・小林（人事）、佐藤（開発）、高橋（営業）

システム移行、第3四半期予算ともに承認され、来月から実施に移ることで決定した。
小林さんはシステム移行の資料作成を5月12日までに仕上げる。
2件同時の承認となったため、実施の優先順位は次回までに整理することにしました。休日開催でしたが5名揃い、一通り議題を消化できました。
次回は来週25日。

###0007###
9月5日（土）　業務効率化定例
出席：中村（人事）、高橋（営業）、佐藤（法務）、山本（経営企画）

新製品ローンチは承認され、来月から実施に移る。業務効率化そのものについては次回会議まで保留とする。
高橋さんはシステム移行のベンダー選定を9月14日までに。佐藤さんは採用計画の社内調整を9月16日までに。中村さんは第3四半期予算の社内調整を9月12日までに進める。
いずれも今月中の対応となるため、進捗は随時共有することにしました。
次回は10月5日。

###0008###
1月3日（土）　システム移行定例
出席：山本（人事）、山本（営業）、中村（開発）、鈴木（人事）

顧客満足度調査は次回会議まで保留とする。年始のため関係者が揃わず、判断は次回以降に持ち越した。
中村さんは業務効率化の資料作成を1月9日までに、要件定義を2月10日までに。第3四半期予算の進捗レポートもあわせて取りまとめる。山本さんは新製品ローンチのベンダー選定を担当する。
中村さんに3件集中しているため、分担は次回相談することに。年始の稼働状況が読めないため、次回日程は未定です。

###0009###
2月22日（日）　システム移行定例
出席：佐藤・高橋・加藤（マーケティング）、加藤（営業）、山本（法務）

顧客満足度調査の予算は100万円、システム移行の予算は300万円でそれぞれ確定した。ただし顧客満足度調査の内容そのものについては再検討とすることになり、設問設計から見直す。
佐藤さんは第3四半期予算の進捗レポートを取りまとめる。
予算は確定したが中身は再検討という形になったため、経緯を議事録に明記しておきます。加藤さんが2名いる点は部署名で区別します。
次回は1ヶ月後の3月24日。

###0010###
5月5日（火）　海外展開定例
出席：高橋（経営企画）、伊藤（開発）、中村（マーケティング）

採用計画は承認され、来月から実施に移ることで決定した。新製品ローンチの予算は1200万円で確定。
中村さんは顧客満足度調査の要件定義を5月21日までに、伊藤さんはセキュリティ監査の資料作成を5月11日までに進める。
最後にもう一度、採用計画は承認・来月実施という結論を全員で確認して終了しました。連休中の開催のため参加者は3名にとどまり、次回日程は未定です。

###0011###
6月25日（木）　採用計画定例
出席：鈴木・佐藤（法務）、中村・山本（マーケティング）、田中（営業）

新製品ローンチは担当をマーケティング部へ移管することで決定した。マーケティングから2名参加していたため、受け入れ体制もその場で確認できた。
鈴木さんは海外展開の見積もり取得を今月末の30日までに、システム移行の要件定義を7月11日までに整理する。
移管に伴う資料の引き継ぎ範囲は、次回までに詰めることにしました。
次回は7月25日。

###0012###
7月20日（月）　採用計画定例
出席：鈴木（開発）、鈴木（法務）、山本（営業）、渡辺（人事）

セキュリティ監査は承認され、来月から実施に移る。顧客満足度調査とシステム移行についてはいずれも再検討とすることになった。
山本さんは新製品ローンチのベンダー選定を8月6日までに。鈴木さんは顧客満足度調査の資料作成を8月9日までに、海外展開の要件定義を8月17日までに進める。
再検討としつつ資料作成は継続する形になるため、どこまで進めるかの線引きを確認しました。鈴木さんが2名いる点は部署名を併記します。次回日程は未定。

###0013###
5月6日（水）　顧客満足度調査定例
出席：佐藤（マーケティング）、山本（経営企画）、渡辺（人事）

第3四半期予算は担当をマーケティング部へ移管することで決定。顧客満足度調査については再検討とし、新製品ローンチは次回会議まで保留とする。
渡辺さんは海外展開の要件定義を5月13日までに、顧客満足度調査の要件定義を5月16日までに整理する。
連休明けということもあり、今回は決定事項が少なめでした。
次回は5月20日。

###0014###
6月13日（土）　顧客満足度調査定例
出席：高橋（人事）、山本（法務）、鈴木（マーケティング）、加藤（経営企画）

新製品ローンチは担当を経営企画部へ移管することで決定。セキュリティ監査については再検討とする。
加藤さんはシステム移行の要件定義を7月14日までに。鈴木さんは第3四半期予算の要件定義を6月26日までに。高橋さんは海外展開の進捗レポートを7月26日までに提出する。
移管を受けた加藤さんに要件定義が乗る形になるため、着手時期は個別に調整することにしました。
次回は来週20日。

###0015###
11月10日（火）　システム移行定例
出席：鈴木（営業）、田中（開発）、中村（マーケティング）、佐藤（人事）

システム移行は担当を人事部へ移管することで決定した。ただし内容そのものについては次回会議まで保留とし、着手の判断は移管後に行う。
田中さんはシステム移行の社内調整を11月19日までに進める。
移管と保留が同時になったため、佐藤さんには準備だけ先行してもらう形で合意しました。
次回は12月10日。

###0016###
9月15日（火）　新製品ローンチ定例
出席：佐藤・鈴木（人事）、小林（マーケティング）、加藤（経営企画）

セキュリティ監査は担当を人事部へ移管することで決定。業務効率化については再検討とする。
鈴木さんは業務効率化の進捗レポートを10月5日までに提出し、顧客満足度調査の資料作成もあわせて担当する。
再検討としながら進捗レポートは出す形になるため、報告の粒度だけ確認しておきました。
次回は9月29日。

###0017###
9月3日（木）　採用計画定例
出席：高橋（人事）、高橋（営業）、鈴木（法務）、山本（人事）

業務効率化は担当を営業部へ移管することで決定。セキュリティ監査については再検討とする。
高橋さん（営業）は業務効率化の社内調整を10月12日までに進める。
高橋さんが2名いるため、担当の記載には部署名を併記します。移管先がその場にいたため、引き継ぎはスムーズに決まりました。
次回は9月17日。

###0018###
7月28日（火）　セキュリティ監査定例
出席：中村（開発）、中村（営業）、伊藤（人事）、山本（経営企画）

新製品ローンチの担当について、当初はマーケティング部へ移管する方向で議論が進んだが、契約面の整理が必要という指摘を受け、最終的に法務部へ移管することで決着した。採用計画は次回会議まで保留とする。
中村さんはセキュリティ監査の見積もり取得を8月24日までに、新製品ローンチの資料作成を8月30日までに、採用計画の社内調整もあわせて担当する。山本さんはシステム移行の見積もり取得を進める。
移管先が途中で変わった経緯は議事録に残しておきます。次回日程は未定。

###0019###
2月25日（水）　新製品ローンチ定例
出席：高橋・鈴木（開発）、田中（マーケティング）

第3四半期予算は次回会議まで保留とする。前提となる数字が固まっていないため、今回は結論を出さず。
第3四半期予算のベンダー選定については、高橋さんが3月23日までに一次候補を、鈴木さんが4月9日までに別の観点から候補を洗い出す。高橋さんはシステム移行の社内調整も3月24日までに。田中さんは新製品ローンチの進捗レポートを3月6日までに提出する。
候補が2系統になるため、最終的な突き合わせは次回に行います。
次回は3月27日。

###0020###
10月26日（月）　第3四半期予算定例
出席：渡辺（マーケティング）、渡辺（営業）、山本（法務）

セキュリティ監査は承認され、来月から実施に移る。新製品ローンチの予算は500万円で確定。採用計画は次回会議まで保留とする。
渡辺さん（営業）は業務効率化の見積もり取得を11月3日までに、海外展開の見積もり取得を12月5日までに進める。山本さんは業務効率化の社内調整を12月2日までに。
渡辺さんが2名いるため、議事録では部署名を併記します。
次回は11月25日。
""")

In [ ]:
status()

In [ ]:
ingest("""
###0021###
10月22日（木）　セキュリティ監査定例
出席：田中（人事）、渡辺（開発）、渡辺（経営企画）、中村（開発）

新製品ローンチの予算は1200万円で確定。システム移行は承認され来月から実施に移るとともに、担当を法務部へ移管することで決定した。
渡辺さん（経営企画）は顧客満足度調査の資料作成を今週26日までに、海外展開の見積もり取得を11月9日までに、顧客満足度調査の見積もり取得を11月11日までに進める。中村さんは海外展開の社内調整を11月16日までに。
渡辺さんに3件集中しているため、分担は次回相談することに。移管先の法務部が不在のため、次回日程は法務側の都合を確認してから決めます。

###0022###
4月11日（土）　第3四半期予算定例
出席：高橋（開発）、中村・加藤（経営企画）、加藤（マーケティング）

顧客満足度調査については再検討とすることになった。設問の設計から見直す必要があるという指摘が複数から出たため。
高橋さんは顧客満足度調査のベンダー選定を5月18日までに。加藤さん（マーケティング）は採用計画の見積もり取得を5月19日までに進める。
再検討としつつベンダー選定は先行させる形になるため、前提だけ確認しました。加藤さんが2名いる点は部署名で区別します。次回日程は未定。

###0023###
1月19日（月）　顧客満足度調査定例
出席：中村（人事）、中村（営業）、渡辺（開発）、田中（経営企画）、佐藤（人事）

採用計画は承認され、来月から実施に移る。第3四半期予算は100万円で確定。セキュリティ監査は担当をマーケティング部へ移管することで決定した。
田中さんは業務効率化の社内調整を2月15日までに。佐藤さんは新製品ローンチの見積もり取得を2月18日までに進める。
移管先のマーケティング部が不参加のため、引き継ぎは別途行います。中村さんが2名いる点は部署名を併記します。
次回は来週26日。

###0024###
4月21日（火）　システム移行定例
出席：佐藤・高橋（開発）、小林・鈴木（営業）、山本（人事）

業務効率化は担当を営業部へ移管することで決定。顧客満足度調査については再検討とし、第3四半期予算は次回会議まで保留とする。
小林さんは採用計画の資料作成を今月28日までに、ベンダー選定を5月28日までに進める。
営業部から2名参加していたため、移管の受け入れはその場で確認できました。
次回は5月21日。

###0025###
5月15日（金）　第3四半期予算定例
出席：伊藤（法務）、山本（経営企画）、小林・佐藤（マーケティング）、中村・田中（人事）

採用計画は承認され、来月から実施に移る。顧客満足度調査の予算は50万円で確定。新製品ローンチについては再検討とする。
田中さんはセキュリティ監査の進捗レポートを5月29日までに提出。中村さんは海外展開の社内調整を5月23日までに。佐藤さんはセキュリティ監査の見積もり取得を6月18日までに。小林さんは採用計画の資料作成を6月29日までに仕上げる。
6名参加で議題も多かったため、次回日程は改めて調整することにしました。

###0026###
2月11日（水）　顧客満足度調査定例
出席：佐藤（人事）、小林（人事）、小林（営業）、山本（営業）、高橋（マーケティング）

業務効率化は担当を営業部へ移管することで決定。顧客満足度調査については再検討とする。
高橋さんは新製品ローンチの見積もり取得を2月24日までに、資料作成を3月17日までに。佐藤さんは業務効率化の資料作成を2月28日までに進める。
祝日開催となりましたが、5名揃って一通り議題を消化できました。小林さんが2名いる点は部署名で区別します。
次回は来週18日。

###0027###
8月23日（日）　顧客満足度調査定例
出席：山本・鈴木（人事）、中村（開発）、中村・高橋（マーケティング）

顧客満足度調査の予算は50万円、業務効率化の予算は100万円でそれぞれ確定した。第3四半期予算については再検討とする。
鈴木さんは採用計画の社内調整を9月19日までに、海外展開の資料作成を9月21日までに、セキュリティ監査の進捗レポートを9月30日までに提出する。高橋さんはシステム移行の要件定義を9月7日までに整理する。
鈴木さんに3件寄っているため、負荷は次回確認することに。中村さんが2名いる点は部署名を併記します。
次回は来週30日。

###0028###
9月19日（土）　海外展開定例
出席：中村（営業）、小林（人事）、田中（法務）

採用計画は次回会議まで保留とする。人員計画の前提が固まっていないため、判断は次回以降に持ち越した。
小林さんは第3四半期予算の要件定義を10月17日までに。田中さんは新製品ローンチの進捗レポートを10月23日までに提出し、海外展開の社内調整を10月26日までに進める。
休日開催で3名のみとなったため、次回日程は改めて全員の予定を確認してから決めます。

###0029###
11月22日（日）　顧客満足度調査定例
出席：佐藤（人事）、佐藤（法務）、鈴木（法務）

新製品ローンチは承認され、来月から実施に移ることで決定した。海外展開の予算は300万円で確定。
佐藤さんは採用計画の社内調整を担当。鈴木さんはセキュリティ監査の進捗レポートを取りまとめる。
休日開催のため3名にとどまりましたが、決定事項は明確に整理できました。佐藤さんが2名いる点は部署名で区別します。
次回は12月6日。

###0030###
10月5日（月）　システム移行定例
出席：伊藤（人事）、伊藤（法務）、田中・佐藤（人事）、小林（開発）

顧客満足度調査は担当を法務部へ移管することで決定。業務効率化の予算は300万円で確定した。
佐藤さんはセキュリティ監査のベンダー選定を10月14日までに。田中さんは業務効率化の進捗レポートを10月11日までに提出。伊藤さん（法務）は顧客満足度調査の要件定義を10月26日までに、社内調整を11月19日までに進める。
次回会議は来週12日と近いため、その時点での進捗のみ共有する形とします。伊藤さんが2名いる点は部署名を併記します。

###0031###
10月26日（月）　海外展開定例
出席：高橋（マーケティング）、伊藤・鈴木・佐藤（法務）

顧客満足度調査は承認され、来月から実施に移る。海外展開は担当を営業部へ移管することで決定し、予算は500万円で確定した。
鈴木さんは顧客満足度調査の社内調整を11月14日までに進め、海外展開の進捗レポートもあわせて取りまとめる。
法務から3名の参加となり、移管に伴う契約面の整理に時間を使いました。営業部は不在のため、引き継ぎは別途行います。
次回は11月9日。

###0032###
2月5日（木）　システム移行定例
出席：小林・田中・中村（経営企画）、小林（営業）、佐藤（人事）、伊藤（マーケティング）

海外展開は承認され、来月から実施に移る。第3四半期予算は100万円で確定した。
伊藤さんは顧客満足度調査の要件定義を3月9日までに整理する。
6名参加でしたが決定事項は2件にとどまり、比較的短時間で終了しました。小林さんが2名いる点は部署名で区別します。
次回は来週12日。

###0033###
1月2日（金）　顧客満足度調査定例
出席：佐藤（営業）、高橋（マーケティング）、鈴木・小林（人事）、中村（経営企画）、加藤（法務）

セキュリティ監査については再検討とすることになり、あわせて担当を営業部へ移管することで決定した。システム移行の予算は500万円で確定。
小林さんは海外展開の要件定義を1月11日までに。中村さんは新製品ローンチの見積もり取得を2月15日までに進める。
年始早々の開催となりましたが、6名揃って一通り議題を消化できました。
次回は来週9日。

###0034###
8月3日（月）　システム移行定例
出席：田中（経営企画）、高橋・加藤（開発）

システム移行は担当を開発部へ移管することで決定した。開発部から2名参加していたため、受け入れ体制もその場で確認できた。
高橋さんは新製品ローンチの進捗レポートを、田中さんは顧客満足度調査のベンダー選定を、いずれも8月15日までに進める。
お盆前に一区切りつけたいという意向があり、期限を揃える形にしました。次回日程は休暇明けに調整します。

###0035###
1月5日（月）　新製品ローンチ定例
出席：高橋・山本（マーケティング）、渡辺（営業）、鈴木（経営企画）、田中（開発）

業務効率化は承認され、来月から実施に移ることで決定した。
山本さんは新製品ローンチの進捗レポートを1月22日までに提出する。
年始最初の会議ということもあり、冒頭は各部の今期方針の共有から入りました。決定事項は1件のみでしたが、方向性の確認はできています。
次回は2月4日。

###0036###
1月2日（金）　業務効率化定例
出席：山本（法務）、佐藤（人事）、佐藤（経営企画）、佐藤（開発）、中村（営業）、加藤（開発）

海外展開、顧客満足度調査ともに担当をマーケティング部へ移管することで決定した。
佐藤さん（経営企画）は海外展開の社内調整を1月19日までに、システム移行の要件定義を2月9日までに、第3四半期予算の要件定義もあわせて担当する。中村さんはセキュリティ監査の見積もり取得を1月11日までに進める。
年始の開催でしたが6名集まりました。佐藤さんが3名いるため、議事録では部署名を必ず併記します。移管先のマーケティング部は不在のため、引き継ぎは別途。
次回は1月16日。

###0037###
3月22日（日）　業務効率化定例
出席：佐藤（営業）、佐藤（人事）、佐藤（法務）

採用計画の予算は1200万円、セキュリティ監査の予算は50万円でそれぞれ確定した。金額の差が大きいが、優先度に応じた配分という整理。海外展開は担当を人事部へ移管することで決定。
システム移行の進捗レポートについては、佐藤さん（営業）が4月11日までに一次分を、佐藤さん（人事）が4月30日までに最終分をそれぞれ提出する。海外展開の資料作成は佐藤さん（法務）が4月8日までに。
偶然にも参加者全員が佐藤姓となり、担当の確認に時間がかかりました。
次回は4月21日。

###0038###
10月22日（木）　新製品ローンチ定例
出席：田中・鈴木（開発）、伊藤（法務）、伊藤（経営企画）、渡辺（営業）

第3四半期予算については、当初は次回会議まで保留とする方向で議論が進んだが、後半で追加の試算が共有されたことで流れが変わり、最終的に承認・来月から実施することで決着した。セキュリティ監査は次回会議まで保留とする。
田中さんは第3四半期予算の進捗レポートを11月14日までに提出し、新製品ローンチの進捗レポートもあわせて取りまとめる。
判断が途中で変わった経緯は議事録に残しておきます。伊藤さんが2名いる点は部署名で区別します。
次回は11月21日。

###0039###
7月16日（木）　業務効率化定例
出席：伊藤（開発）、小林（営業）、小林（法務）、渡辺・加藤（人事）、佐藤（経営企画）

システム移行、新製品ローンチともに承認され、来月から実施に移ることで決定した。
伊藤さんは海外展開の見積もり取得を8月4日までに。小林さん（法務）はセキュリティ監査の要件定義を担当する。
2件同時の実施となるため、リソースの配分は次回までに整理することにしました。お盆をはさむため次回日程は未定です。小林さんが2名いる点は部署名を併記します。

###0040###
9月23日（水）　システム移行定例
出席：佐藤・田中・加藤（人事）、佐藤（経営企画）

業務効率化は担当を営業部へ移管することで決定。セキュリティ監査と新製品ローンチはいずれも次回会議まで保留とする。
田中さんは第3四半期予算の資料作成を10月6日までに、海外展開の要件定義を10月15日までに、採用計画の見積もり取得を10月18日までに進める。佐藤さん（人事）はセキュリティ監査の社内調整を10月11日までに。
田中さんに3件寄っているため、分担は次回相談することに。祝日開催で人事部中心の構成となり、次回日程は他部署の予定を確認してから決めます。
""")

In [ ]:
status()

In [ ]:
ingest("""
###0041###
5月7日（木）　採用計画定例
出席：佐藤（開発）、中村（人事）、中村（法務）、加藤（マーケティング）

海外展開は担当を開発部へ移管することで決定。顧客満足度調査の予算は1200万円で確定した。第3四半期予算は次回会議まで保留とする。
業務効率化の社内調整については、佐藤さんが5月29日までに関係部署への一次説明を、6月11日までに合意形成までを完了させる。第3四半期予算の見積もり取得も佐藤さんが担当する。
佐藤さんに集中しているため、負荷は次回確認することに。中村さんが2名いる点は部署名を併記します。
次回は6月6日。

###0042###
11月7日（土）　海外展開定例
出席：小林（開発）、田中（人事）、佐藤（経営企画）、佐藤（マーケティング）、鈴木（営業）、高橋（法務）

セキュリティ監査の予算は500万円で確定した。採用計画については次回会議まで保留とする。
佐藤さん（経営企画）は第3四半期予算の見積もり取得を12月21日までに進める。
6名参加でしたが、休日開催ということもあり短時間で切り上げました。年末にかけて各自の稼働が読みづらいため、次回日程は改めて調整します。佐藤さんが2名いる点は部署名で区別します。

###0043###
11月13日（金）　顧客満足度調査定例
出席：加藤（開発）、加藤（法務）、渡辺（営業）、鈴木（経営企画）、伊藤（法務）

顧客満足度調査は承認され、来月から実施に移ることで決定した。採用計画は次回会議まで保留とするが、予算枠は1200万円で確定している。
渡辺さんは顧客満足度調査の見積もり取得を11月17日までに、システム移行の社内調整を12月7日までに。加藤さん（開発）は顧客満足度調査の資料作成を11月25日までに、新製品ローンチの要件定義を12月20日までに進める。
保留としつつ予算だけ先行して確定した経緯は議事録に残しておきます。
次回は12月13日。

###0044###
3月20日（金）　セキュリティ監査定例
出席：鈴木・加藤（開発）、中村（営業）、山本（経営企画）

業務効率化については再検討とするが、予算枠は100万円で確定した。システム移行は次回会議まで保留とする。
顧客満足度調査のベンダー選定については、鈴木さんが4月30日までに候補を絞り込み、山本さんも別の観点から並行して洗い出す。加藤さんはセキュリティ監査の社内調整を4月10日までに、鈴木さんは第3四半期予算の進捗レポートを4月15日までに提出する。
候補が2系統になるため、突き合わせは次回に行います。
次回は来週27日。

###0045###
7月3日（金）　システム移行定例
出席：渡辺（営業）、渡辺（開発）、山本（営業）、山本（マーケティング）

第3四半期予算は300万円で確定した。当初案より抑えた形だが、まずはこの範囲で進める。
山本さん（マーケティング）は新製品ローンチの資料作成を来週月曜までに。渡辺さん（営業）は採用計画の社内調整を担当する。
同姓の参加者が2組という構成になり、議事録では部署名を必ず併記します。次回日程は各自の予定を確認してから調整することにしました。

###0046###
3月18日（水）　セキュリティ監査定例
出席：伊藤（開発）、伊藤（営業）、田中（開発）

業務効率化は担当を営業部へ移管することで決定。新製品ローンチは次回会議まで保留とする。
田中さんは新製品ローンチの社内調整を4月8日までに、採用計画の社内調整を4月26日までに進める。
保留としつつ社内調整は先行させる形になるため、どこまで進めるかの線引きを確認しました。伊藤さんが2名いる点は部署名で区別します。
次回は4月17日。

###0047###
6月3日（水）　新製品ローンチ定例
出席：中村（営業）、山本・田中・佐藤（経営企画）、伊藤・高橋（法務）

海外展開は承認され、来月から実施に移ることで決定し、予算は100万円で確定した。業務効率化は次回会議まで保留とする。
山本さんは顧客満足度調査の資料作成を7月5日までに仕上げる。
6名参加で議論は活発でしたが、アクションは1件に絞る形になりました。実施に向けた体制は別途整理するため、次回日程は追って調整します。

###0048###
4月2日（木）　海外展開定例
出席：小林（人事）、伊藤・渡辺（営業）、高橋（経営企画）

顧客満足度調査の予算は50万円、採用計画の予算は500万円でそれぞれ確定した。セキュリティ監査は次回会議まで保留とする。
高橋さんは顧客満足度調査の見積もり取得を4月10日までに、業務効率化のベンダー選定を次回会議までに完了させ、採用計画の進捗レポートもあわせて取りまとめる。渡辺さんは第3四半期予算の見積もり取得を5月8日までに進める。
高橋さんに3件寄っているため、分担は次回相談することにしました。
次回は5月2日。

###0049###
6月20日（土）　セキュリティ監査定例
出席：田中・鈴木（営業）、鈴木（マーケティング）、山本（経営企画）、小林（法務）

新製品ローンチは担当を開発部へ移管することで決定したうえで、内容そのものは次回会議まで保留とする。海外展開については再検討とする。
鈴木さん（営業）は第3四半期予算の社内調整を7月16日までに。山本さんは新製品ローンチの資料作成を7月8日までに。小林さんは第3四半期予算の進捗レポートを7月17日までに提出し、セキュリティ監査の見積もり取得もあわせて担当する。
移管先の開発部が不在のため、引き継ぎは別途行います。鈴木さんが2名いる点は部署名で区別します。
次回は7月20日。

###0050###
8月7日（金）　顧客満足度調査定例
出席：高橋・鈴木（営業）、中村（営業）、中村（開発）、渡辺（法務）

顧客満足度調査は担当を開発部へ移管することで決定した。中村さん（開発）が窓口になる。
鈴木さんは第3四半期予算のベンダー選定を9月11日までに。中村さん（営業）は新製品ローンチの社内調整を8月25日までに進める。
お盆前ということもあり、着手は今週中を目安とすることにしました。中村さんが2名いる点は部署名を併記します。
次回は8月21日。

###0051###
8月26日（水）　採用計画定例
出席：佐藤（経営企画）、山本（法務）、中村（マーケティング）

業務効率化は承認され、来月から実施に移る。第3四半期予算は次回会議まで保留とする。
山本さんは採用計画の資料作成を9月1日までに仕上げ、システム移行の見積もり取得もあわせて担当する。佐藤さんは採用計画の進捗レポートを取りまとめる。
少人数の開催となりましたが、決定事項は明確に整理できました。
次回は9月25日。

###0052###
5月28日（木）　第3四半期予算定例
出席：高橋・鈴木（法務）、小林（経営企画）、小林（営業）、伊藤（マーケティング）

第3四半期予算は承認され、来月から実施に移る。海外展開は担当を営業部へ移管することで決定。採用計画については再検討とする。
伊藤さんは顧客満足度調査の見積もり取得を7月4日までに、新製品ローンチの社内調整を7月6日までに進める。鈴木さんはセキュリティ監査の要件定義を担当する。
小林さんが2名いるため、議事録では部署名を併記します。
次回は6月27日。

###0053###
1月6日（火）　業務効率化定例
出席：中村（人事）、渡辺（人事）、高橋（経営企画）、加藤（開発）、渡辺（法務）

第3四半期予算は300万円で確定した。年始最初の会議のため、決定事項は1件にとどまった。
高橋さんは海外展開の見積もり取得を1月19日までに進め、システム移行の社内調整もあわせて担当する。渡辺さん（法務）は業務効率化の進捗レポートを2月9日までに提出し、セキュリティ監査の見積もり取得も担当する。
渡辺さんが2名いる点は部署名で区別します。
次回は1月20日。

###0054###
1月3日（土）　セキュリティ監査定例
出席：高橋（開発）、加藤（経営企画）、加藤（法務）、田中（営業）、伊藤（人事）、中村（経営企画）

第3四半期予算、セキュリティ監査ともに承認され、来月から実施に移ることで決定した。海外展開の予算は500万円で確定。
加藤さん（経営企画）はセキュリティ監査の資料作成を1月24日までに、第3四半期予算の見積もり取得を2月10日までに進める。
年始早々の開催でしたが6名揃い、今期の方針まで一通り確認できました。加藤さんが2名いる点は部署名を併記します。
次回は1月17日。

###0055###
6月26日（金）　海外展開定例
出席：鈴木（開発）、田中（開発）、田中（営業）、加藤・山本（マーケティング）

業務効率化の予算は1200万円で確定した。採用計画についても予算枠は500万円で確定したが、内容そのものは再検討とすることになり、着手の判断は次回以降に持ち越す。
田中さん（営業）は業務効率化の進捗レポートを7月31日までに提出。鈴木さんは顧客満足度調査の進捗レポートを7月25日までに。山本さんは海外展開の資料作成を担当する。
予算だけ先に確定した経緯は議事録に残しておきます。田中さんが2名いる点は部署名で区別します。
次回は7月10日。

###0056###
11月26日（木）　システム移行定例
出席：加藤（人事）、鈴木（マーケティング）

新製品ローンチについては再検討とすることになった。前提となる市場データが更新されるため、それを待ってからの議論とする。
鈴木さんは海外展開の社内調整を担当する。
2名のみの開催となったため、決定事項は関係者に別途共有します。年末にかけて日程が読めないため、次回は改めて調整することにしました。

###0057###
12月7日（月）　セキュリティ監査定例
出席：田中・渡辺（法務）、鈴木・高橋・中村（マーケティング）、山本（経営企画）

海外展開は次回会議まで保留とする。年内の判断は難しいという整理で一致した。
中村さんは顧客満足度調査のベンダー選定を12月22日までに。山本さんは顧客満足度調査の見積もり取得を、高橋さんは業務効率化の要件定義をそれぞれ担当する。
年末進行で各自の稼働が厳しいため、年内対応は1件に絞る形にしました。
次回は年明けの1月6日。

###0058###
12月9日（水）　顧客満足度調査定例
出席：加藤（法務）、小林（人事）、山本（経営企画）

新製品ローンチの予算は300万円で確定した。
山本さんは顧客満足度調査の要件定義を年明け1月9日までに整理する。
年末で参加者が3名にとどまりましたが、予算の確定はできました。要件定義は年をまたぐため、着手時期だけ確認しておきました。
次回は来週16日。

###0059###
9月9日（水）　顧客満足度調査定例
出席：佐藤（マーケティング）、加藤・小林（開発）、高橋（営業）

新製品ローンチについては再検討とすることになった。業務効率化の予算は50万円で確定。
加藤さんはセキュリティ監査のベンダー選定と業務効率化の社内調整をいずれも9月14日までに、第3四半期予算の資料作成を10月23日までに進める。高橋さんはシステム移行の社内調整を10月12日までに。
会議の最後に、新製品ローンチは再検討という結論を改めて全員で確認して終了しました。
次回は9月23日。

###0060###
1月1日（木）　顧客満足度調査定例
出席：渡辺（経営企画）、中村（法務）、小林（マーケティング）

海外展開は承認され、来月から実施に移ることで決定した。業務効率化については再検討とする。
中村さんは海外展開の資料作成を1月15日までに、セキュリティ監査のベンダー選定を1月16日までに進める。渡辺さんは第3四半期予算の社内調整を1月9日までに、システム移行の見積もり取得を2月3日までに。
元日の開催となり参加者は3名でしたが、今期の方針だけ先に固めておく形としました。
次回は来週8日。
""")

In [ ]:
status()

In [ ]:
ingest("""
###0061###
8月25日（火）　顧客満足度調査定例
出席：伊藤（マーケティング）、伊藤（法務）、渡辺（法務）、加藤・高橋（営業）

業務効率化は承認され、来月から実施に移る。海外展開の予算は1200万円で確定した。顧客満足度調査そのものは次回会議まで保留とする。
高橋さんは業務効率化のベンダー選定を9月27日までに。加藤さんはシステム移行の社内調整を10月2日までに進める。
お盆明けで揃わないメンバーもいたため、保留案件は次回に持ち越しました。伊藤さんが2名いる点は部署名で区別します。
次回は9月24日。

###0062###
2月17日（火）　業務効率化定例
出席：佐藤・小林・高橋（マーケティング）、佐藤（営業）、小林（開発）

業務効率化の予算は50万円で確定した。まずは小さく始めて、効果を見てから追加を検討する方針。
佐藤さん（マーケティング）は業務効率化の資料作成を3月23日までに。小林さん（開発）は業務効率化の社内調整を3月24日までに進める。
同姓の参加者が2組いるため、議事録では部署名を必ず併記します。年度末にかけて各自の稼働が読みづらいため、次回日程は改めて調整することにしました。

###0063###
8月22日（土）　顧客満足度調査定例
出席：山本（経営企画）、田中（法務）、高橋（マーケティング）

セキュリティ監査は承認され、来月から実施に移ることで決定した。
山本さんは業務効率化の資料作成を今週28日までに仕上げる。
お盆明けの休日開催となり3名にとどまりましたが、承認までは進められました。実施体制の詳細は次回で詰めます。
次回は来週29日。

###0064###
3月25日（水）　セキュリティ監査定例
出席：山本（人事）、佐藤（法務）、小林（開発）、渡辺（営業）

セキュリティ監査は担当を開発部へ移管することで決定した。小林さんが窓口になる。
渡辺さんは業務効率化の見積もり取得を今月末の29日までに。小林さんは業務効率化のベンダー選定を4月11日までに進める。
年度末の移管となるため、引き継ぎは新年度に入ってから改めて整理することにしました。次回日程は未定です。

###0065###
4月24日（金）　第3四半期予算定例
出席：山本（マーケティング）、田中（開発）、高橋（営業）

新製品ローンチは担当をマーケティング部へ移管することで決定した。山本さんが引き取る形になる。
田中さんはセキュリティ監査の資料作成を担当する。
新年度の体制がまだ固まりきっていないため、決定事項は移管の1件にとどまりました。次回の日程は体制確定後に調整します。

###0066###
4月25日（土）　新製品ローンチ定例
出席：小林・加藤（経営企画）、高橋（法務）、加藤（マーケティング）

システム移行は担当を経営企画部へ移管することで決定し、予算は300万円で確定。新製品ローンチの予算は1200万円で確定した。
小林さんは業務効率化の進捗レポートを5月25日までに提出。加藤さん（経営企画）はセキュリティ監査の要件定義を6月8日までに整理する。
移管を受けた経営企画部に2件乗る形になるため、負荷は次回確認することに。加藤さんが2名いる点は部署名で区別します。
次回は5月9日。

###0067###
3月10日（火）　セキュリティ監査定例
出席：小林・鈴木（人事）、高橋（開発）、伊藤（法務）

セキュリティ監査は担当をマーケティング部へ移管することで決定した。マーケティング部は本日不参加のため、引き継ぎは個別に行う。
小林さんは海外展開のベンダー選定を3月29日までに、システム移行の資料作成を4月5日までに進める。
年度末をまたぐ移管となるため、資料の引き継ぎ範囲は次回までに整理することにしました。
次回は4月9日。

###0068###
10月20日（火）　新製品ローンチ定例
出席：高橋（マーケティング）、小林・中村（開発）

業務効率化は承認され、来月から実施に移る。新製品ローンチの予算は100万円で確定した。
小林さんは海外展開の進捗レポートを11月7日までに提出し、海外展開の要件定義を11月26日までに整理する。
予算が想定より抑えられた形になったため、スコープの調整は次回で議論することにしました。
次回は11月19日。

###0069###
8月15日（土）　第3四半期予算定例
出席：渡辺（開発）、渡辺（経営企画）、加藤（開発）、山本・鈴木（営業）、伊藤（マーケティング）

セキュリティ監査は担当を開発部へ、システム移行は担当をマーケティング部へそれぞれ移管することで決定。海外展開については再検討とする。
鈴木さんは新製品ローンチのベンダー選定を9月18日までに。加藤さんは業務効率化の社内調整を、渡辺さん（開発）は採用計画の見積もり取得をそれぞれ担当する。
お盆期間の開催でしたが6名揃いました。移管が2件重なったため、引き継ぎの順序は別途整理します。渡辺さんが2名いる点は部署名を併記。次回日程は休暇明けに調整します。

###0070###
2月27日（金）　新製品ローンチ定例
出席：山本（開発）、高橋・鈴木（営業）

業務効率化は担当を営業部へ移管することで決定した。営業部から2名参加していたため、受け入れはその場で確認できた。
山本さんはセキュリティ監査の資料作成と進捗レポートを、いずれも来週火曜の3月3日までに揃える。採用計画の資料作成もあわせて担当する。鈴木さんは顧客満足度調査の要件定義を進める。
山本さんに3件寄っているため、次回までに分担を見直すことに。次回日程は未定です。

###0071###
9月9日（水）　海外展開定例
出席：山本（営業）、渡辺（人事）、中村（マーケティング）、高橋（法務）、高橋（経営企画）

新製品ローンチは承認され、来月から実施に移ることで決定した。セキュリティ監査は次回会議まで保留とする。
山本さんは海外展開のベンダー選定を9月14日までに。中村さんは業務効率化のベンダー選定を9月26日までに。高橋さん（経営企画）は第3四半期予算の見積もり取得を10月23日までに進め、顧客満足度調査の資料作成もあわせて担当する。
会議の最後に、新製品ローンチは承認・来月実施という結論を改めて全員で確認しました。高橋さんが2名いる点は部署名で区別します。
次回は10月9日。

###0072###
3月25日（水）　セキュリティ監査定例
出席：伊藤（開発）、加藤（開発）、加藤（人事）

海外展開は担当を法務部へ移管することで決定。新製品ローンチの予算は50万円で確定した。第3四半期予算については再検討とする。
加藤さん（開発）はセキュリティ監査の進捗レポートを4月18日までに提出し、業務効率化の見積もり取得もあわせて担当する。
年度末の開催で3名にとどまりました。移管先の法務部が不在のため、引き継ぎは新年度に入ってから行います。次回日程は未定。

###0073###
9月13日（日）　セキュリティ監査定例
出席：加藤（法務）、加藤（経営企画）、山本（営業）、高橋（法務）

顧客満足度調査は承認され、来月から実施に移ることで決定した。
加藤さん（法務）は第3四半期予算の見積もり取得を10月6日までに進め、業務効率化の見積もり取得もあわせて担当する。
休日開催のため4名にとどまりましたが、承認は問題なく通りました。加藤さんが2名いる点は部署名を併記します。
次回は来週20日。

###0074###
3月3日（火）　採用計画定例
出席：中村・小林・渡辺・佐藤（マーケティング）、山本・田中（営業）

システム移行は担当をマーケティング部へ移管することで決定した。セキュリティ監査は次回会議まで保留とするが、予算枠は50万円で確定している。
山本さんは顧客満足度調査のベンダー選定を3月16日までに。小林さんは第3四半期予算のベンダー選定を担当する。
マーケティングから4名参加という構成になり、移管の受け入れ体制はその場で固められました。
次回は3月17日。

###0075###
5月16日（土）　第3四半期予算定例
出席：伊藤・加藤（開発）、中村（マーケティング）、中村（営業）、鈴木（経営企画）

海外展開は次回会議まで保留とする。為替の前提が定まっていないため、判断は次回以降に持ち越した。
中村さん（営業）は海外展開の社内調整を今月末の31日までに。鈴木さんは業務効率化の見積もり取得を6月8日までに。加藤さんは採用計画の要件定義を6月15日までに整理する。
保留としつつ社内調整は進める形になるため、線引きを確認しました。中村さんが2名いる点は部署名で区別します。
次回は5月30日。

###0076###
7月9日（木）　セキュリティ監査定例
出席：佐藤・高橋（マーケティング）、小林（開発）、小林（営業）、中村（法務）

顧客満足度調査は次回会議まで保留とする。設問の設計を見直したいという意見があり、今回は結論を出さず。
小林さん（営業）は新製品ローンチの進捗レポートを8月7日までに、第3四半期予算の社内調整を次回会議までに完了させ、システム移行の見積もり取得もあわせて担当する。中村さんはセキュリティ監査の進捗レポートを8月11日までに提出する。
小林さんに3件寄っているため、分担は次回相談することに。小林さんが2名いる点は部署名を併記します。
次回は8月8日。

###0077###
10月21日（水）　第3四半期予算定例
出席：高橋（法務）、渡辺（開発）、加藤（営業）

第3四半期予算は承認され、来月から実施に移る。システム移行は担当を経営企画部へ移管することで決定。海外展開は次回会議まで保留とする。
渡辺さんは採用計画の資料作成を10月29日までに、業務効率化のベンダー選定を11月22日までに、顧客満足度調査の資料作成を11月23日までに、採用計画の要件定義を11月30日までに進める。
アクションが渡辺さんに4件集中しているため、次回までに必ず分担を見直すことにしました。
次回は11月4日。

###0078###
12月23日（水）　業務効率化定例
出席：佐藤（経営企画）、佐藤（営業）、佐藤（法務）、田中・渡辺（法務）

採用計画については再検討とすることになった。年内に結論を出すのは難しいという判断。
田中さんはセキュリティ監査のベンダー選定を年明け1月24日までに。渡辺さんはセキュリティ監査の要件定義とシステム移行の進捗レポートをそれぞれ担当する。
年末最終週の開催となり、年内対応は見送る形にしました。佐藤さんが3名いるため、議事録では部署名を必ず併記します。
次回は年明けの1月6日。

###0079###
1月23日（金）　システム移行定例
出席：伊藤（法務）、佐藤（マーケティング）、鈴木（経営企画）

業務効率化については再検討とする。海外展開は次回会議まで保留とする。今回は決定に至った案件がなく、持ち帰りが中心となった。
鈴木さんは採用計画の見積もり取得を2月6日までに、業務効率化の見積もり取得を2月17日までに進め、顧客満足度調査の要件定義もあわせて担当する。伊藤さんは第3四半期予算の見積もり取得を2月3日までに。
鈴木さんに3件寄っているため、負荷は次回確認することにしました。
次回は2月22日。

###0080###
3月19日（木）　セキュリティ監査定例
出席：鈴木（人事）、鈴木（マーケティング）、鈴木（経営企画）、小林（法務）、伊藤（開発）

新製品ローンチは担当を営業部へ移管することで決定。採用計画は次回会議まで保留とする。
伊藤さんは業務効率化の社内調整を4月22日までに。鈴木さん（経営企画）も業務効率化の社内調整を別途担当し、システム移行の見積もり取得もあわせて進める。小林さんは新製品ローンチの進捗レポートを取りまとめる。
鈴木さんが3名いるため、担当の確認に時間がかかりました。以降は部署名を必ず併記します。
次回は4月18日。
""")

In [ ]:
status()

In [ ]:
ingest("""
###0081###
2月9日（月）　海外展開定例
出席：中村（マーケティング）、高橋（人事）、佐藤（法務）

システム移行は承認され、来月から実施に移る。海外展開の予算は50万円で確定した。顧客満足度調査については再検討とする。
佐藤さんは新製品ローンチのベンダー選定を2月25日までに。高橋さんは採用計画の進捗レポートを3月1日までに提出。中村さんはセキュリティ監査の要件定義を3月23日までに整理する。
海外展開の予算が想定より小さくなったため、スコープの見直しは次回に持ち越しました。
次回は2週間後の23日。

###0082###
5月19日（火）　顧客満足度調査定例
出席：鈴木（人事）、鈴木（法務）、田中（経営企画）

採用計画については再検討とすることになった。前提となる人員計画が固まっていないため、いったん戻す形とした。
鈴木さん（人事）は新製品ローンチの進捗レポートを6月11日までに提出し、顧客満足度調査の要件定義を6月27日までに整理する。田中さんは海外展開の要件定義を6月8日までに、第3四半期予算の資料作成を6月19日までに進める。
少人数でしたがアクションは多めになりました。鈴木さんが2名いる点は部署名で区別します。次回日程は未定。

###0083###
8月21日（金）　海外展開定例
出席：小林（人事）、高橋（人事）

採用計画は担当を人事部へ移管することで決定。顧客満足度調査の予算は300万円で確定した。
高橋さんは第3四半期予算の資料作成を来週27日までに仕上げ、業務効率化のベンダー選定もあわせて担当する。小林さんはシステム移行の社内調整を進める。
お盆明けで参加者が2名にとどまりましたが、移管先である人事部が揃っていたため決定はスムーズでした。
次回は9月4日。

###0084###
3月28日（土）　顧客満足度調査定例
出席：渡辺・佐藤（経営企画）、鈴木（営業）

第3四半期予算は承認され、来月から実施に移る。システム移行の予算は500万円で確定した。
渡辺さんは海外展開の資料作成を今月末の31日までに。佐藤さんは採用計画の要件定義を4月27日までに整理する。鈴木さんはシステム移行の進捗レポートと第3四半期予算の社内調整をそれぞれ担当する。
年度末の駆け込みとなりましたが、必要な決定は済ませられました。
次回は4月4日。

###0085###
6月12日（金）　顧客満足度調査定例
出席：山本（開発）、鈴木（営業）、小林・田中（マーケティング）

業務効率化、採用計画についてはいずれも再検討とすることになった。システム移行は次回会議まで保留とする。
小林さんはシステム移行のベンダー選定を7月5日までに、セキュリティ監査の要件定義を7月22日までに進める。
今回は決定に至った案件がなく、持ち帰りが中心となりました。論点の整理は次回までに各自で行うことにしています。
次回は来週19日。

###0086###
9月20日（日）　顧客満足度調査定例
出席：伊藤（開発）、小林（開発）、小林（マーケティング）

顧客満足度調査、セキュリティ監査、第3四半期予算のいずれも次回会議まで保留とする。連休中で関係者が揃わず、判断は次回以降に持ち越した。
伊藤さんはシステム移行の見積もり取得を9月23日までに、第3四半期予算の社内調整を10月4日までに。小林さん（開発）はセキュリティ監査の社内調整を10月1日までに進める。
保留が3件重なったため、次回は判断を優先することにしました。小林さんが2名いる点は部署名で区別します。
次回は10月4日。

###0087###
9月10日（木）　セキュリティ監査定例
出席：山本（営業）、伊藤（営業）、小林（マーケティング）、小林（開発）、加藤（開発）

海外展開は担当をマーケティング部へ移管することで決定した。小林さん（マーケティング）が窓口になる。
山本さんはシステム移行の要件定義を10月10日までに整理する。
移管に伴う引き継ぎ範囲の整理に時間を使ったため、他の議題は次回に回しました。小林さんが2名いる点は部署名を併記します。次回日程は未定です。

###0088###
7月4日（土）　業務効率化定例
出席：伊藤・田中（法務）、鈴木（営業）

採用計画は承認され、来月から実施に移ることで決定した。
鈴木さんはセキュリティ監査の見積もり取得を今週8日までに、第3四半期予算の要件定義を7月23日までに進める。田中さんは顧客満足度調査のベンダー選定を担当する。
休日開催で3名にとどまりましたが、承認は問題なく通りました。
次回は7月18日。

###0089###
1月16日（金）　採用計画定例
出席：鈴木（経営企画）、佐藤（マーケティング）、佐藤（開発）、小林・田中（営業）

業務効率化は承認され、来月から実施に移ることで決定した。
佐藤さん（開発）は海外展開の要件定義を2月14日までに整理し、セキュリティ監査の資料作成もあわせて担当する。
年始の体制確認に時間を使ったため、決定事項は1件にとどまりました。佐藤さんが2名いる点は部署名で区別します。
次回は2月15日。

###0090###
6月4日（木）　新製品ローンチ定例
出席：高橋（マーケティング）、田中・佐藤（人事）、伊藤（経営企画）

業務効率化は承認され、来月から実施に移る。顧客満足度調査については再検討とすることになった。
田中さんはシステム移行の社内調整を7月14日までに進め、新製品ローンチの社内調整もあわせて担当する。
アクションが田中さんに集中しているため、分担は次回相談することにしました。
次回は1ヶ月後の7月4日。

###0091###
12月1日（火）　海外展開定例
出席：山本（法務）、伊藤（営業）、高橋（マーケティング）

業務効率化は次回会議まで保留とする。年内の判断は難しいという整理で一致した。
セキュリティ監査のベンダー選定については、高橋さんと山本さんがそれぞれ別の候補先を当たる形で進める。高橋さんは第3四半期予算の要件定義を年明け1月2日までに整理する。
候補が2系統になるため、突き合わせは次回に行います。
次回は来週8日。

###0092###
7月22日（水）　採用計画定例
出席：伊藤・渡辺・中村（経営企画）、渡辺（人事）

システム移行は承認され、来月から実施に移ることで決定した。
中村さんは業務効率化のベンダー選定を担当する。
経営企画から3名参加という構成になり、予算面の確認に時間を使いました。決定事項は1件にとどまりましたが、方向性は固まっています。渡辺さんが2名いる点は部署名で区別します。
次回は来週29日。

###0093###
9月12日（土）　システム移行定例
出席：伊藤（人事）、伊藤（営業）、伊藤（法務）、佐藤（開発）、加藤（営業）、高橋（経営企画）

システム移行は承認され、来月から実施に移ることで決定した。大きな異論はなく、比較的スムーズにまとまった。
伊藤さん（法務）は新製品ローンチのベンダー選定を10月1日までに進める。
休日開催でしたが6名揃いました。伊藤さんが3名いるため、議事録では部署名を必ず併記します。
次回は10月12日。

###0094###
12月18日（金）　システム移行定例
出席：中村・鈴木・田中（営業）

システム移行については再検討とすることになった。年内に結論を出すのは難しく、前提から見直すという判断。
中村さんは新製品ローンチの資料作成を年明け1月6日までに仕上げる。
年末進行で営業部のみの参加となりました。決定事項は関係者に別途共有します。次回日程は年明けに改めて調整することにしました。

###0095###
11月15日（日）　採用計画定例
出席：渡辺（営業）、田中（人事）、田中（開発）

採用計画については再検討とすることになった。セキュリティ監査は次回会議まで保留とする。
システム移行の進捗レポートについては、田中さん（人事）が11月20日までに一次分を提出し、その後の追加分も引き続き取りまとめる。採用計画の社内調整も田中さんが担当する。渡辺さんは業務効率化の社内調整を進める。
会議の終わりに、セキュリティ監査は保留という結論を改めて確認しました。田中さんが2名いる点は部署名で区別します。
次回は12月15日。

###0096###
1月13日（火）　海外展開定例
出席：小林（人事）、小林（経営企画）、渡辺（マーケティング）、山本（経営企画）、鈴木（営業）

顧客満足度調査は承認され、来月から実施に移ることで決定した。
渡辺さんは第3四半期予算のベンダー選定を1月30日までに、新製品ローンチの進捗レポートを2月2日までに、採用計画の見積もり取得を2月9日までに進める。
渡辺さんに3件集中しているため、分担は次回相談することにしました。年始で各自の予定が読みづらいため、次回日程は改めて調整します。小林さんが2名いる点は部署名で区別。

###0097###
6月26日（金）　海外展開定例
出席：渡辺（マーケティング）、加藤（開発）

第3四半期予算は承認され、来月から実施に移る。顧客満足度調査は担当を人事部へ移管することで決定した。
渡辺さんは海外展開のベンダー選定を7月11日までに進める。加藤さんは新製品ローンチの社内調整を担当する。
2名のみの開催となったため、決定事項は関係者に別途共有します。移管先の人事部にも個別に引き継ぎを行います。
次回は7月10日。

###0098###
1月1日（木）　採用計画定例
出席：佐藤・山本（マーケティング）、渡辺（営業）、渡辺（法務）

顧客満足度調査の予算は300万円で確定した。
渡辺さん（営業）は業務効率化の要件定義を今月末の31日までに整理し、海外展開の進捗レポートもあわせて取りまとめる。
元日の開催となりましたが、今期の予算枠だけ先に固めておく形としました。渡辺さんが2名いる点は部署名で区別します。
次回は来週8日。

###0099###
7月10日（金）　顧客満足度調査定例
出席：鈴木・田中（法務）、加藤（人事）、加藤（営業）

海外展開は担当を開発部へ移管することで決定した。開発部は本日不参加のため、引き継ぎは個別に行う。
田中さんは第3四半期予算の見積もり取得を7月22日までに進める。
法務中心の構成となり、移管に伴う契約面の整理に時間を使いました。加藤さんが2名いる点は部署名を併記します。
次回は7月24日。

###0100###
3月21日（土）　第3四半期予算定例
出席：中村（営業）、伊藤（営業）、伊藤（開発）、高橋（法務）、佐藤（経営企画）

新製品ローンチは承認され、来月から実施に移ることで決定した。採用計画と海外展開はいずれも次回会議まで保留とする。
伊藤さん（開発）はシステム移行の要件定義を4月29日までに整理する。
年度末の祝日開催でしたが5名揃いました。保留となった2件は新年度に入ってから改めて議論します。伊藤さんが2名いる点は部署名で区別します。
次回は来週28日。
""")

In [ ]:
import json, random

# 抽驗 5 筆，確認沒錯位
for i in random.sample(list(pairs.keys()), 5):
    p = pairs[i]
    print(f"[{i:04d}] {p['note'][:12]} | {p['json']['date']}")

In [ ]:
print(os.path.exists(PAIR_PATH), len(pairs))

In [18]:
!mkdir -p /content/data
!cp "{BASE}records.json" /content/data/
!cp "{BASE}pairs.json" /content/data/
!ls -lh /content/data

total 396K
-rw------- 1 root root 154K Aug 12 06:22 pairs.json
-rw------- 1 root root 237K Aug 12 06:22 records.json
